# City of Agents - Google Colab Setup

Quick setup guide for running City of Agents in Google Colab.

## 1. Clone Repository

Set the branch name to match your current development branch.

In [1]:
import os

REPO_URL = "https://github.com/mohameddhameem/City-of-Agents.git"
REPO_NAME = "City-of-Agents"
BRANCH = "feat/updated-folder-structure"

if not os.path.exists(REPO_NAME):
    !git clone --branch {BRANCH} {REPO_URL}
else:
    print(f"Directory {REPO_NAME} already exists")

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

Cloning into 'City-of-Agents'...
remote: Enumerating objects: 546, done.
remote: Counting objects: 100% (546/546), done.
remote: Compressing objects: 100% (508/508), done.
remote: Total 546 (delta 53), reused 502 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (546/546), 11.82 MiB | 17.75 MiB/s, done.
Resolving deltas: 100% (53/53), done.
Working directory: /content/City-of-Agents


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Access Shared Data

To access the shared Google Drive folder:

1. In Google Drive, locate the shared folder and select "Add shortcut to Drive"
2. Save the shortcut to "My Drive"
3. The code below will access it from `/content/drive/MyDrive/CPG`

In [3]:
shared_folder_path = '/content/drive/MyDrive/CPG'

if os.path.exists(shared_folder_path):
    contents = os.listdir(shared_folder_path)
    print(f"Folder found: {contents}")
else:
    print(f"Folder not found at {shared_folder_path}")

Folder found: ['data_split', 'raw10', 'processed_hetero', 'processed_homo', 'raw']


## 3. Install Package

In [4]:
import sys
import subprocess

print("Installing dependencies...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."],
                       capture_output=True, text=True)
if result.returncode == 0:
    print("Dependencies installed successfully")
else:
    print("Installation error:", result.stderr if result.stderr else result.stdout)

Installing dependencies...
Dependencies installed successfully


## 4. Test Imports

In [5]:
import sys
import os

try:
    import city_of_agents
    from city_of_agents import ast2pyg, joern
    from city_of_agents.builders import city
    from city_of_agents.utils import CPGHomoDataset, CPGHeteroDataset
    print("All imports successful")
except Exception as e:
    print(f"Import error: {e}")
    print(f"Python version: {sys.version}")
    print(f"Working directory: {os.getcwd()}")

Import error: No module named 'city_of_agents'
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working directory: /content/City-of-Agents


## 5. Generate Sample Data

In [6]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 150

df = pd.DataFrame({
    "Task_ID": np.random.randint(1, 11, n),
    "Model": np.random.choice(["GPT-4o", "Claude 3.5", "Llama 3"], n),
    "Language": np.random.choice(["Python", "Java"], n),
    "Token_Count": np.clip(np.random.normal(120, 30, n), 1, None),
    "Max_Depth": np.clip(np.random.normal(8, 2, n), 1, None)
})

df.to_csv("simulation_data.csv", index=False)
print(f"Generated {len(df)} samples")
df.head()

Generated 150 samples


,Task_ID,Model,Language,Token_Count,Max_Depth
0,7,Llama 3,Java,115.928479,4.970850
1,4,Llama 3,Python,96.194849,9.359319
2,8,Claude 3.5,Java,108.946500,7.754423
3,5,Llama 3,Java,124.716056,9.297786
4,7,GPT-4o,Java,116.535798,9.560550


## 6. Visualize

In [9]:
import plotly.graph_objects as go

df = pd.read_csv("simulation_data.csv")
colors = {"GPT-4o": "red", "Claude 3.5": "blue", "Llama 3": "green"}
x_map = {"GPT-4o": 0, "Claude 3.5": 4, "Llama 3": 8}

np.random.seed(42)
df["x"] = df["Model"].map(x_map) + np.random.uniform(-0.3, 0.3, len(df))
df["y"] = df["Task_ID"] + np.random.uniform(-0.2, 0.2, len(df))

fig = go.Figure()
for model, color in colors.items():
    sub = df[df["Model"] == model]
    fig.add_trace(go.Scatter3d(
        x=sub["x"], y=sub["y"], z=sub["Max_Depth"],
        mode="markers", marker=dict(size=8, color=color),
        name=model
    ))

fig.update_layout(
    title="City of Agents",
    scene=dict(xaxis_title="Model", yaxis_title="Task", zaxis_title="Depth"),
    height=600
)
fig.show()

## Next Steps

You can now explore the package modules:

- `ast2pyg`: Convert AST to PyG graphs
- `joern`: Run Joern to generate CPG graphs
- `CPGHomoDataset` / `CPGHeteroDataset`: Load graph datasets

The dataset contains 55 problems with 2,623 code samples.